# §3c at Scale: Risk–Coverage on N=1200, With Confidence Intervals

## The problem

§3c's headline — **95.8% top-decile precision** — rests on **23 of 24 calls**.

| | value |
|---|---|
| point estimate | 23/24 = **95.8%** |
| Wilson 95% CI | **[79.8%, 99.3%]** |
| CI width | **19.5 percentage points** |

A reviewer will flag this immediately, and they will be right to: the interval is consistent with
true precision anywhere from "barely better than the 72.1% base rate" to "essentially perfect."
The number is not wrong — it is just not yet *supported*.

## What this notebook fixes

**Three things, and the second matters as much as the first.**

1. **N = 1200** instead of 400. Top-decile coverage becomes ~120 sequences and ~72 predicted-collapse
   calls, tightening the CI to roughly **[88.5%, 98.6%]** — about 10 points instead of 19.5.

   | pool N | top-decile covered | collapse calls | expected 95% CI |
   |---|---|---|---|
   | 400 (§3c) | 40 | 24 | [79.8%, 99.3%] |
   | **1200 (this)** | **120** | **~72** | **[88.5%, 98.6%]** |
   | 1600 | 160 | ~96 | [89.8%, 98.4%] |

2. **A Wilson CI on every cell of the risk–coverage table.** §3c reported bare point estimates.
   Even at N=400 the honest presentation was "95.8% [79.8, 99.3]", and reporting intervals is a
   fix a reviewer will value independently of the sample size.

3. **10 repeats of stratified 10-fold CV**, not one. §3c used a single `cross_val_predict` pass, so
   its number also carries fold-assignment noise on top of sampling noise. Reporting the mean and
   spread across repeats separates the two.

**Plus a diagnostic that makes the case for the rerun explicit:** bootstrap 200 sub-samples of
N=400 out of the 1200 and recompute top-decile precision on each. **That shows directly how far
§3c's number could have wandered by luck of the draw** — a far more convincing argument than
quoting a CI, and it costs no GPU time.

## What is deliberately held fixed

Everything else matches `08` so this is a scale-up, not a new experiment: unsteered ProtGPT2, real
UniProt calibration prefixes, `max_length=50` tokens (natural length, **not** the fixed-length
regime of `54`–`58` — §3a/§3c were free-length and this must stay comparable), pLDDT < 60 collapse
label, teacher-forced re-encoding, `hidden_states[30]` mean-pooled, StandardScaler → PCA-20 →
RandomForest.

The `hidden_states` vs hook convention is immaterial here (§1q-C: cosine +0.9985 on ProtGPT2), but
`08`'s convention is kept so the AUCs are directly comparable.

## Pre-registered readings

| outcome | reading |
|---|---|
| precision ≈ 95%, CI excludes the 72% base rate | §3c confirmed and now properly supported. Cite the CI, not the bare number. |
| precision materially lower (say 80–88%) | §3c's 95.8% was an N=24 artifact. **Report the new number and say so** — the selective-prediction claim survives in weaker form, since even 85% beats the 72.1% base rate. |
| precision at or near the base rate | The abstention claim does not survive scaling. Retract §3c. |

Note the third row is a real possibility and the notebook is built to report it cleanly.

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**. Expect **~2–2.5 hours**
(1200 generations, 1200 folds at natural length, 1200 activation passes; the classifier work is
seconds).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from scipy.stats import fisher_exact, mannwhitneyu

torch.manual_seed(2026)
np.random.seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ProtGPT2's own natural v_L norm from notebook 03. Used ONLY to define the anchor and to report
# how far the old absolute-norm runs were from matched -- never to scale anything in this notebook.
REFERENCE_NORM = 583.998

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())
print("Model under test: nferruz/ProtGPT2")


Setup complete. CUDA available: True
Model under test: nferruz/ProtGPT2


In [2]:
# --- The same common probe set as 34/38/40/41/47: real UniProt fragments. Using one shared probe
#     set across every repair notebook is what makes their alpha_rel values directly comparable
#     rather than each notebook being its own island. ---

UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

FALLBACK_SEQS = [
    "NLYIQWLKDGGPSSGRPPPS",
    "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF",
    "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING -- Kaggle's Internet toggle is likely OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!! Falling back to hardcoded reference sequences so the run can proceed, but the probe")
    print("!! set will then differ from 34/38/40/41/47 -- say so if you log this result.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(FALLBACK_SEQS)]

def build_probe_set(reference_seqs, n_probes=40, frag_len=50, seed=7):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    probes = []
    for i in range(n_probes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        L = min(frag_len, len(seq))
        start = rng.randint(0, max(1, len(seq) - L + 1))
        probes.append(seq[start:start + L])
    return probes

def build_prefix_pool(reference_seqs, n_prefixes, min_len=10, max_len=15, seed=11):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    prefixes = []
    for i in range(n_prefixes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        plen = rng.randint(min_len, max_len + 1)
        start = rng.randint(0, max(1, len(seq) - plen))
        prefixes.append(seq[start:start + plen])
    return prefixes

probe_seqs = build_probe_set(reference_seqs, n_probes=40)
print(f"\nBuilt {len(probe_seqs)} common probe fragments from {len(reference_seqs)} source proteins.")

def get_layer_module(model, path, idx):
    obj = model
    for part in path.split("."):
        obj = getattr(obj, part)
    return obj[idx]

def measure_resid_norm_mod(model, layer_module, tokenize_fn, seqs):
    # Mean per-position residual-stream norm at one layer. This is the ‖h‖ that alpha_rel
    # divides by, and measuring it per-layer is the entire point of this notebook.
    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, (tuple, list)) else out
        captured["h"] = h.detach()
    handle = layer_module.register_forward_hook(hook)
    vals = []
    try:
        for seq in seqs:
            enc = tokenize_fn(seq)
            captured.clear()
            with torch.no_grad():
                model(**enc)
            if "h" in captured:
                vals.append(captured["h"].float().norm(dim=-1).mean().item())
    finally:
        handle.remove()
    return float(np.mean(vals)) if vals else float("nan")

print("Probe set and residual-norm measurement ready.")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues

Built 40 common probe fragments from 12 source proteins.
Probe set and residual-norm measurement ready.


In [3]:
# --- Scoring + ESMFold. Identical to 24/26/27/41, including the length-aware OOM retry added in
#     39/40 and the fold_ok tracking added after the §1l collapse-metric hole. ---
ALPHABET_SIZE = 20

def h_norm(seq):
    if not seq:
        return 0.0
    counts = collections.Counter(seq)
    total = len(seq)
    ent = -sum((c / total) * math.log2(c / total) for c in counts.values())
    return ent / math.log2(ALPHABET_SIZE)

def distinct_n(seq, n):
    if len(seq) < n:
        return 1.0
    grams = [seq[i:i + n] for i in range(len(seq) - n + 1)]
    return len(set(grams)) / len(grams)

def homopolymer_runs(seq):
    if not seq:
        return []
    runs, run_len = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return runs

def r_hpoly(seq, k=4):
    if not seq:
        return 1.0
    T = len(seq)
    penalty = sum(l for l in homopolymer_runs(seq) if l >= k)
    return max(0.0, 1.0 - penalty / T)

def repetition_score(seq):
    return float(np.mean([h_norm(seq), distinct_n(seq, 2), distinct_n(seq, 3), r_hpoly(seq)]))

def utility_score(plddt, ptm):
    return float(np.mean([plddt / 100.0, ptm]))

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluatorPTM:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq, max_len=300):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0, False
        working = cleaned[:max_len] if len(cleaned) > max_len else cleaned
        try:
            inputs = self.tokenizer([working], return_tensors="pt", add_special_tokens=False).to(self.device)
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
            return plddt, ptm, True
        except RuntimeError:
            clear_gpu()
        half = max(10, len(working) // 2)
        if half < len(working):
            try:
                inputs = self.tokenizer([working[:half]], return_tensors="pt", add_special_tokens=False).to(self.device)
                with torch.no_grad():
                    out = self.model(**inputs)
                raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
                plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
                ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
                return plddt, ptm, True
            except RuntimeError:
                clear_gpu()
        return 0.0, 0.0, False

def fold_records_ptm(records, evaluator):
    for r in records:
        plddt, ptm, fold_ok = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["fold_ok"] = fold_ok
        r["collapse"] = int(0.0 < plddt < 60.0)
        r["repetition_score"] = repetition_score(r["sequence"])
        r["utility_score"] = utility_score(plddt, ptm) if plddt > 0 else 0.0
    return records

print("Scoring functions and length-aware ESMFold evaluator ready.")


Scoring functions and length-aware ESMFold evaluator ready.


In [4]:
# --- Generate the pool. Same recipe as 08, just 3x bigger. ---

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score

N_POOL = 1200
LAYER = 30
PCA_DIM = 20
N_REPEATS = 10
COVERAGES = [0.10, 0.20, 0.30, 0.50, 0.70, 1.00]

print(f"Loading ProtGPT2 on {device}...")
tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()
PAD_ID = tokenizer.eos_token_id
print(f"{plm_model.config.num_hidden_layers} layers; using layer {LAYER} activations (as in 08).")

def clean_aa(text):
    return "".join(a for a in text.replace(" ", "") if a in VALID_AA)

print(f"\n=== Generating N={N_POOL} unsteered candidates (max_length=50 tokens, as in 08) ===")
torch.manual_seed(505)
prefixes = build_prefix_pool(reference_seqs, n_prefixes=N_POOL, seed=11)
records = []
for i, p in enumerate(prefixes):
    inputs = tokenizer(p, return_tensors="pt").to(device)
    with torch.no_grad():
        out_ids = plm_model.generate(**inputs, max_length=50, do_sample=True,
                                     temperature=1.2, pad_token_id=PAD_ID)
    seq = tokenizer.decode(out_ids[0], skip_special_tokens=True).replace(" ", "")
    records.append({"prompt": p, "sequence": seq, "clean": clean_aa(seq),
                    "entropy": calculate_entropy(seq)})
    if (i + 1) % 200 == 0:
        print(f"  ...{i + 1}/{N_POOL}")
clear_gpu()

lens = [len(r["clean"]) for r in records]
print(f"\nusable length: mean {np.mean(lens):.1f}, median {np.median(lens):.0f}, max {max(lens)}")
print("(08's pool had mean ~67; 49's control 66.8 -- close values confirm the same regime.)")


Loading ProtGPT2 on cuda...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


36 layers; using layer 30 activations (as in 08).

=== Generating N=1200 unsteered candidates (max_length=50 tokens, as in 08) ===
  ...200/1200
  ...400/1200
  ...600/1200
  ...800/1200
  ...1000/1200
  ...1200/1200

usable length: mean 59.3, median 39, max 322
(08's pool had mean ~67; 49's control 66.8 -- close values confirm the same regime.)


In [5]:
# --- Extract layer-30 activations by teacher-forced re-encoding, BEFORE folding, so ProtGPT2 and
#     ESMFold never need to be resident at the same time. ---

print(f"Extracting layer-{LAYER} activations for {len(records)} sequences...")
feats = []
for i, r in enumerate(records):
    s = r["clean"] if len(r["clean"]) >= 5 else "MKT"
    enc = tokenizer(s, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = plm_model(**enc, output_hidden_states=True)
    feats.append(out.hidden_states[LAYER].float().mean(dim=1).squeeze(0).cpu().numpy())
    if (i + 1) % 300 == 0:
        print(f"  ...{i + 1}/{len(records)}")
X = np.vstack(feats)
print(f"activation matrix: {X.shape}")

del plm_model
clear_gpu()
print("ProtGPT2 freed.")


Extracting layer-30 activations for 1200 sequences...
  ...300/1200
  ...600/1200
  ...900/1200
  ...1200/1200
activation matrix: (1200, 1280)
ProtGPT2 freed.


In [6]:
# --- Fold everything. This is the expensive cell. ---

evaluator = StructuralEvaluatorPTM()
print(f"Folding {len(records)} sequences...")
for i, r in enumerate(records):
    plddt, ptm, ok = evaluator.fold_one(r["sequence"])
    r["plddt"] = plddt
    r["ptm"] = ptm
    r["fold_ok"] = ok
    r["collapse"] = int(0.0 < plddt < 60.0)
    r["collapse70"] = int(0.0 < plddt < 70.0)
    if (i + 1) % 100 == 0:
        done = [q for q in records[:i + 1] if q["fold_ok"]]
        print(f"  ...{i + 1}/{len(records)}  running collapse "
              f"{np.mean([q['collapse'] for q in done]):.1%}")
del evaluator
clear_gpu()

ok_mask = np.array([r["fold_ok"] for r in records])
y = np.array([r["collapse"] for r in records])[ok_mask]
Xf = X[ok_mask]
ent = np.array([r["entropy"] for r in records])[ok_mask].reshape(-1, 1)
print(f"\n{ok_mask.sum()}/{len(records)} folded successfully.")
print(f"collapse rate (pLDDT<60): {y.mean():.1%}   (08 recorded 53.2% at N=400)")
print(f"mean pLDDT: {np.mean([r['plddt'] for r in records if r['fold_ok']]):.2f}")
if ok_mask.sum() < len(records):
    print(f"!! {len(records) - ok_mask.sum()} sequences failed to fold and are EXCLUDED "
          f"(not counted as successes -- the §1l trap).")


Loading ESMFold...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding 1200 sequences...
  ...100/1200  running collapse 57.0%
  ...200/1200  running collapse 57.0%
  ...300/1200  running collapse 53.3%
  ...400/1200  running collapse 53.2%
  ...500/1200  running collapse 52.6%
  ...600/1200  running collapse 52.0%
  ...700/1200  running collapse 52.3%
  ...800/1200  running collapse 52.0%
  ...900/1200  running collapse 51.1%
  ...1000/1200  running collapse 51.1%
  ...1100/1200  running collapse 51.9%
  ...1200/1200  running collapse 52.2%

1200/1200 folded successfully.
collapse rate (pLDDT<60): 52.2%   (08 recorded 53.2% at N=400)
mean pLDDT: 59.32


In [8]:
# --- Out-of-fold probabilities: 10 repeats of stratified 10-fold. ---
# FIX: only apply PCA when there are more features than components. The entropy baseline is a
# single column, and PCA(n_components=20) on 1 feature raises ValueError.
def make_pipe(seed, n_features=None):
    steps = [("scale", StandardScaler())]
    if n_features is None or n_features > PCA_DIM:
        steps.append(("pca", PCA(n_components=PCA_DIM, random_state=seed)))
    steps.append(("rf", RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1)))
    return Pipeline(steps)

oof_reps, auc_reps = [], []
for rep in range(N_REPEATS):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=1000 + rep)
    p = cross_val_predict(make_pipe(1000 + rep, Xf.shape[1]), Xf, y, cv=cv,
                          method="predict_proba")[:, 1]
    oof_reps.append(p)
    auc_reps.append(roc_auc_score(y, p))
    print(f"  repeat {rep + 1:2d}/{N_REPEATS}: OOF AUC {auc_reps[-1]:.4f}")
oof_reps = np.array(oof_reps)

print()
print(f"OOF ROC-AUC over {N_REPEATS} repeats: {np.mean(auc_reps):.4f} +- {np.std(auc_reps):.4f}")
print(f"  range [{min(auc_reps):.4f}, {max(auc_reps):.4f}]")
print(f"  (3c reported 0.767 from a single 10-fold pass; 3a 0.772 +- 0.034 from 20x 70/30 splits)")

ent_auc = [roc_auc_score(y, cross_val_predict(
    make_pipe(2000 + r, ent.shape[1]), ent, y,
    cv=StratifiedKFold(10, shuffle=True, random_state=2000 + r),
    method="predict_proba")[:, 1]) for r in range(3)]
print(f"entropy baseline AUC: {np.mean(ent_auc):.4f}  (3a recorded 0.586)")

rng = np.random.RandomState(7)
perm_auc = []
for r in range(3):
    ys = rng.permutation(y)
    p = cross_val_predict(make_pipe(3000 + r, Xf.shape[1]), Xf, ys,
                          cv=StratifiedKFold(10, shuffle=True, random_state=3000 + r),
                          method="predict_proba")[:, 1]
    perm_auc.append(roc_auc_score(ys, p))
print(f"permutation control AUC: {np.mean(perm_auc):.4f} +- {np.std(perm_auc):.4f}  "
      f"(3a recorded 0.496 +- 0.044)")

  repeat  1/10: OOF AUC 0.8072
  repeat  2/10: OOF AUC 0.8187
  repeat  3/10: OOF AUC 0.8128
  repeat  4/10: OOF AUC 0.8231
  repeat  5/10: OOF AUC 0.8166
  repeat  6/10: OOF AUC 0.8190
  repeat  7/10: OOF AUC 0.8159
  repeat  8/10: OOF AUC 0.8231
  repeat  9/10: OOF AUC 0.8259
  repeat 10/10: OOF AUC 0.8229

OOF ROC-AUC over 10 repeats: 0.8185 +- 0.0054
  range [0.8072, 0.8259]
  (3c reported 0.767 from a single 10-fold pass; 3a 0.772 +- 0.034 from 20x 70/30 splits)
entropy baseline AUC: 0.5091  (3a recorded 0.586)
permutation control AUC: 0.5040 +- 0.0129  (3a recorded 0.496 +- 0.044)


In [9]:
# --- The risk-coverage table, with Wilson CIs. Definition matches 3c exactly:
#       confidence = max(p, 1-p); sort descending; keep the top X% ("covered");
#       risk       = error rate among covered at threshold 0.5;
#       precision  = among covered sequences PREDICTED to collapse, the fraction that truly did.
# ---

def wilson(k, n, z=1.959963985):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return (max(0.0, c - h), min(1.0, c + h))

def risk_coverage(p, y_true, cov):
    conf = np.maximum(p, 1 - p)
    order = np.argsort(-conf)
    k = max(1, int(round(cov * len(p))))
    idx = order[:k]
    pred = (p[idx] > 0.5).astype(int)
    truth = y_true[idx]
    risk = float(np.mean(pred != truth))
    call = pred == 1
    n_call = int(call.sum())
    n_hit = int((truth[call] == 1).sum())
    prec = n_hit / n_call if n_call else float("nan")
    return k, risk, prec, n_hit, n_call

print(f"{'Coverage':>9}{'N cov':>7}{'Risk':>8}{'Precision':>11}{'n':>10}{'Wilson 95% CI':>22}"
      f"{'across-repeat sd':>19}")
print("-" * 88)
rc_rows = []
for cov in COVERAGES:
    per_rep = [risk_coverage(oof_reps[r], y, cov) for r in range(N_REPEATS)]
    k = per_rep[0][0]
    risk = float(np.mean([t[1] for t in per_rep]))
    prec = float(np.mean([t[2] for t in per_rep]))
    prec_sd = float(np.std([t[2] for t in per_rep]))
    hits = int(round(np.mean([t[3] for t in per_rep])))
    calls = int(round(np.mean([t[4] for t in per_rep])))
    lo, hi = wilson(hits, calls)
    rc_rows.append({"coverage": cov, "n_covered": k, "risk": risk, "precision": prec,
                    "precision_sd_across_repeats": prec_sd, "n_hits": hits, "n_calls": calls,
                    "ci_lo": lo, "ci_hi": hi})
    print(f"{cov:>8.0%}{k:>7}{risk:>8.3f}{prec:>10.1%}{f'{hits}/{calls}':>10}"
          f"{f'[{lo:.1%}, {hi:.1%}]':>22}{prec_sd:>18.3f}")

base = float(y.mean())
print()
print(f"Base rate (collapse prevalence): {base:.1%}. A precision CI whose lower bound clears this")
print("is what makes the abstention claim meaningful -- precision above chance, not just high.")

top = rc_rows[0]
# Build the comparison strings first -- nested same-quote f-strings are a 3.12-only feature and
# have broken generators in this project before (CLAUDE.md sec 5).
now_n = str(top["n_calls"])
now_prec = "{:.1%}".format(top["precision"])
now_ci = "[{:.1%}, {:.1%}]".format(top["ci_lo"], top["ci_hi"])
print()
print("Read against 3c (N=400, single CV pass):")
print("{:16}{:>22}{:>26}".format("", "3c", "this run"))
print("{:16}{:>22}{:>26}".format("top-decile n", "24", now_n))
print("{:16}{:>22}{:>26}".format("precision", "95.8%", now_prec))
print("{:16}{:>22}{:>26}".format("95% CI", "[79.8%, 99.3%]", now_ci))


 Coverage  N cov    Risk  Precision         n         Wilson 95% CI   across-repeat sd
----------------------------------------------------------------------------------------
     10%    120   0.032     98.1%     42/43        [87.9%, 99.6%]             0.017
     20%    240   0.043     95.0%   111/117        [89.3%, 97.6%]             0.017
     30%    360   0.067     92.6%   180/195        [87.7%, 95.3%]             0.013
     50%    600   0.134     85.8%   292/341        [81.5%, 89.0%]             0.007
     70%    840   0.197     79.4%   383/483        [75.5%, 82.7%]             0.008
    100%   1200   0.268     72.6%   489/673        [69.2%, 75.9%]             0.008

Base rate (collapse prevalence): 52.2%. A precision CI whose lower bound clears this
is what makes the abstention claim meaningful -- precision above chance, not just high.

Read against 3c (N=400, single CV pass):
                                    3c                  this run
top-decile n                        24 

In [10]:
# --- The diagnostic that justifies the rerun: how much could 3c's number have wandered?
#     Draw 200 sub-samples of N=400 from this pool and recompute top-decile precision on each.
#     Uses the already-computed OOF probabilities, so no refitting and no GPU. ---

N_SUB, N_DRAWS = 400, 200
rng2 = np.random.RandomState(2026)
sub_prec, sub_calls = [], []
for d in range(N_DRAWS):
    idx = rng2.choice(len(y), N_SUB, replace=False)
    rep = d % N_REPEATS
    _, _, prec, hits, calls = risk_coverage(oof_reps[rep][idx], y[idx], 0.10)
    if calls >= 5:
        sub_prec.append(prec)
        sub_calls.append(calls)

sub_prec = np.array(sub_prec)
print(f"Top-decile precision across {len(sub_prec)} sub-samples of N={N_SUB}:")
print(f"  mean   {sub_prec.mean():.1%}")
print(f"  sd     {sub_prec.std():.1%}")
print(f"  range  [{sub_prec.min():.1%}, {sub_prec.max():.1%}]")
print(f"  5th-95th percentile  [{np.percentile(sub_prec, 5):.1%}, "
      f"{np.percentile(sub_prec, 95):.1%}]")
print(f"  median calls per draw: {int(np.median(sub_calls))}")
print()
print(f"3c drew ONE such sample and got 95.8%. The spread above is how much that single draw could")
print("have differed by luck alone -- with the true value fixed at this pool's estimate. This is")
print("the honest argument for the rerun, and it is worth one sentence in the paper.")
frac_above = float((sub_prec >= 0.958).mean())
print()
print(f"Fraction of N=400 draws that would have reported >= 95.8%: {frac_above:.1%}")


Top-decile precision across 200 sub-samples of N=400:
  mean   97.8%
  sd     3.6%
  range  [82.4%, 100.0%]
  5th-95th percentile  [91.7%, 100.0%]
  median calls per draw: 14

3c drew ONE such sample and got 95.8%. The spread above is how much that single draw could
have differed by luck alone -- with the true value fixed at this pool's estimate. This is
the honest argument for the rerun, and it is worth one sentence in the paper.

Fraction of N=400 draws that would have reported >= 95.8%: 70.5%


In [11]:
# --- Verdict. ---

BAR = "=" * 92
top = rc_rows[0]
print(BAR)
print("VERDICT -- does 3c's selective-prediction claim survive scaling?")
print(BAR)
print(f"  N = {len(y)} folded sequences, base rate {base:.1%}")
print(f"  OOF AUC {np.mean(auc_reps):.3f} +- {np.std(auc_reps):.3f}  "
      f"(3a: 0.772 +- 0.034, 3c: 0.767)")
print("  top-decile precision {:.1%}  [{:.1%}, {:.1%}]  on {} calls".format(
    top["precision"], top["ci_lo"], top["ci_hi"], top["n_calls"]))
print()
clears_base = top["ci_lo"] > base
near_3c = abs(top["precision"] - 0.958) < 0.07
if clears_base and near_3c:
    print("  ==> 3c CONFIRMED AND PROPERLY SUPPORTED. The point estimate holds at 3x the sample")
    print("      size and the CI lower bound ({:.1%}) clears the {:.1%} base rate.".format(
        top["ci_lo"], base))
    print("      Cite the interval, not the bare number. Replace 3c's table with this one.")
elif clears_base:
    print("  ==> THE CLAIM SURVIVES IN WEAKER FORM. Precision is materially below 3c's 95.8%, so")
    print("      that figure was an N=24 artifact -- but the CI still clears the base rate, so")
    print("      selective prediction does work. Report THIS number and state plainly that the")
    print("      earlier one did not survive scaling.")
else:
    print("  ==> THE CLAIM DOES NOT SURVIVE. The precision CI includes the base rate, so")
    print("      restricting to confident predictions buys nothing demonstrable. Retract 3c and")
    print("      report the abstention result as a null. The AUC finding (3a) is unaffected --")
    print("      it is a separate measurement on the same features.")
print(BAR)


VERDICT -- does 3c's selective-prediction claim survive scaling?
  N = 1200 folded sequences, base rate 52.2%
  OOF AUC 0.819 +- 0.005  (3a: 0.772 +- 0.034, 3c: 0.767)
  top-decile precision 98.1%  [87.9%, 99.6%]  on 43 calls

  ==> 3c CONFIRMED AND PROPERLY SUPPORTED. The point estimate holds at 3x the sample
      size and the CI lower bound (87.9%) clears the 52.2% base rate.
      Cite the interval, not the bare number. Replace 3c's table with this one.


In [ ]:
# --- Persist. ---

pd.DataFrame([{
    "idx": i, "prompt": r["prompt"], "sequence": r["sequence"],
    "usable_length": len(r["clean"]), "entropy": r["entropy"],
    "plddt": r["plddt"], "ptm": r["ptm"], "fold_ok": r["fold_ok"],
    "collapse": r["collapse"], "collapse70": r["collapse70"],
} for i, r in enumerate(records)]).to_csv("riskcoverage_n1200_sequences.csv", index=False)

pd.DataFrame(rc_rows).to_csv("riskcoverage_n1200_table.csv", index=False)

pd.DataFrame([{"repeat": r, "oof_auc": auc_reps[r]} for r in range(N_REPEATS)]
             ).to_csv("riskcoverage_n1200_auc_repeats.csv", index=False)

pd.DataFrame([{"draw": i, "top_decile_precision": float(p)} for i, p in enumerate(sub_prec)]
             ).to_csv("riskcoverage_n1200_subsample_n400.csv", index=False)

pd.DataFrame([{
    "n_generated": len(records), "n_folded": int(ok_mask.sum()),
    "base_rate": base, "layer": LAYER, "pca_dim": PCA_DIM, "n_repeats": N_REPEATS,
    "oof_auc_mean": float(np.mean(auc_reps)), "oof_auc_sd": float(np.std(auc_reps)),
    "entropy_baseline_auc": float(np.mean(ent_auc)),
    "permutation_auc": float(np.mean(perm_auc)),
    "top_decile_precision": top["precision"], "top_decile_ci_lo": top["ci_lo"],
    "top_decile_ci_hi": top["ci_hi"], "top_decile_calls": top["n_calls"],
    "subsample400_precision_mean": float(sub_prec.mean()),
    "subsample400_precision_sd": float(sub_prec.std()),
    "subsample400_p05": float(np.percentile(sub_prec, 5)),
    "subsample400_p95": float(np.percentile(sub_prec, 95)),
    "mean_usable_length": float(np.mean(lens)),
    "used_uniprot_fallback": USED_FALLBACK,
}]).to_csv("riskcoverage_n1200_calibration.csv", index=False)

print("Saved:")
for f in ["riskcoverage_n1200_sequences.csv", "riskcoverage_n1200_table.csv",
          "riskcoverage_n1200_auc_repeats.csv", "riskcoverage_n1200_subsample_n400.csv",
          "riskcoverage_n1200_calibration.csv"]:
    print("  " + f)
print()
print("DOWNLOAD THE OUTPUT TAB BEFORE CLOSING THE SESSION.")
print()
print("riskcoverage_n1200_sequences.csv is worth keeping regardless of the verdict: it is a")
print("1200-sequence ProtGPT2 pool with pLDDT labels, three times anything else in the project,")
print("and any future prediction-arm analysis can reuse it without regenerating.")
